# Exploratory Data Analysis (EDA) on Cars Dataset

This notebook explores a cars dataset to understand its structure, quality, and patterns before building a predictive model.

The goal is to:
- load and inspect the data
- identify missing values and anomalies
- analyze distributions and relationships
- summarize key insights that can guide feature engineering and modeling


In [ ]:
# Importing the necessary libraries
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.style.use("seaborn-v0_8-darkgrid")


## 1. Load the dataset

In [ ]:
# Find a CSV file automatically in the current directory
csv_files = glob.glob('*.csv')
csv_files


In [ ]:
# If you know the exact file name, replace the fallback below with that name
candidate_files = [
    'car_data.csv',
    'Cars.csv',
    'cars.csv',
    'cars_dataset.csv',
    'vehicle_data.csv'
]

dataset_path = None
for file in candidate_files:
    if os.path.exists(file):
        dataset_path = file
        break

if dataset_path is None and csv_files:
    dataset_path = csv_files[0]

if dataset_path is None:
    raise FileNotFoundError(
        'No CSV dataset file was found in the current directory. Please update the file name in this notebook.'
    )

df = pd.read_csv(dataset_path)
print(f"Loaded dataset: {dataset_path}")
print(f"Shape: {df.shape}")
df.head()


## 2. Basic data inspection

In [ ]:
# Dataset information
df.info()


In [ ]:
# Descriptive statistics for numeric columns
df.describe(include='all').T


In [ ]:
# Check for missing values
missing = df.isnull().sum()
missing[missing > 0]


In [ ]:
# Check duplicate rows
duplicate_rows = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicate_rows}")
df.drop_duplicates(inplace=True)
print(f"Shape after removing duplicates: {df.shape}")


## 3. Identify key columns and data types

In [ ]:
# Show columns
df.columns.tolist()


In [ ]:
# Print unique value counts for key categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols[:10]:
    print(f'\nColumn: {col}')
    print(df[col].value_counts().head(10))


## 4. Univariate analysis

In [ ]:
# Identify the target column (likely price)
target_col = 'price' if 'price' in df.columns else None
if target_col is None:
    print('Target column not found. Please set the target column manually.')
else:
    plt.figure(figsize=(10, 6))
    sns.histplot(df[target_col], kde=True, bins=30)
    plt.title(f'Distribution of {target_col}')
    plt.xlabel(target_col)
    plt.ylabel('Count')
    plt.show()


In [ ]:
# Numeric columns distribution
numeric_cols = df.select_dtypes(include=['number']).columns
print('Numeric columns:', list(numeric_cols))

for col in numeric_cols[:6]:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], kde=True, bins=25)
    plt.title(f'Distribution of {col}')
    plt.show()


In [ ]:
# Check top brands or makes
if 'make' in df.columns:
    plt.figure(figsize=(12, 6))
    sns.countplot(data=df, x='make', order=df['make'].value_counts().head(10).index, palette='viridis')
    plt.title('Top 10 Car Makes')
    plt.xticks(rotation=45)
    plt.show()


## 5. Bivariate analysis

In [ ]:
# Price vs. other numeric variables
if target_col in df.columns:
    for col in numeric_cols:
        if col == target_col:
            continue
        plt.figure(figsize=(8, 5))
        sns.scatterplot(data=df, x=col, y=target_col)
        plt.title(f'{target_col} vs {col}')
        plt.show()


In [ ]:
# Relationship between categorical and target variables
if target_col in df.columns and 'make' in df.columns:
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df, x='make', y=target_col, order=df['make'].value_counts().head(10).index)
    plt.title(f'{target_col} by Car Make')
    plt.xticks(rotation=45)
    plt.show()


In [ ]:
# Correlation heatmap
corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


## 6. Outlier and quality checks

In [ ]:
# Boxplot of numeric features to detect outliers
for col in numeric_cols[:5]:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x=col)
    plt.title(f'Boxplot of {col}')
    plt.show()


In [ ]:
# Handle extreme outliers only if needed
# Example: remove rows with unrealistic price values
if target_col in df.columns:
    q1 = df[target_col].quantile(0.25)
    q3 = df[target_col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[target_col] < lower) | (df[target_col] > upper)]
    print(f"Potential outliers in {target_col}: {len(outliers)}")


## 7. Feature engineering ideas

In [ ]:
# Example feature engineering steps
# 1. Convert categorical columns to numeric codes if needed
# 2. Create age of the car from year if a 'year' column exists
# 3. Add log transform for skewed price data

if 'year' in df.columns:
    df['car_age'] = 2025 - df['year']
    print(df[['year', 'car_age']].head())

if target_col in df.columns:
    df['log_price'] = np.log1p(df[target_col])
    plt.figure(figsize=(8, 5))
    sns.histplot(df['log_price'], kde=True, bins=30)
    plt.title('Distribution of log(price)')
    plt.show()


## 8. Key insights and summary

From the analysis, we can infer important patterns such as:
- distribution of vehicle price
- relationship between age, mileage, make, and price
- which features show strong correlation with price
- presence of anomalies or missing data

These findings are useful for choosing the right model and improving prediction quality.